# Fresnel Diffraction Simulator

Interactive simulator for a **single slit** and an opaque **straight edge**, based on [`theory and context/Difraccion_Fresnel_Clotoide.pdf`](theory%20and%20context/Difraccion_Fresnel_Clotoide.pdf).

The implementation uses SciPy's normalized Fresnel integrals

$$C(u)=\int_0^u\cos\left(\frac{\pi t^2}{2}\right)dt,\qquad S(u)=\int_0^u\sin\left(\frac{\pi t^2}{2}\right)dt.$$

- Slit: the field is the Cornu-spiral chord between the two aperture-edge coordinates.
- Edge: the field is the chord from the edge coordinate to $(C,S)=(1/2,1/2)$.
- Both plane-wave and finite point-source illumination are available.
- The camera shows the one-dimensional Fresnel solution across the invariant axis, while Profiles and Cornu views expose its quantitative construction.

All physical calculations use metres and the in-medium wavelength $\lambda=\lambda_0/n$. Unlike the Fraunhofer notebook, Fresnel number is an informational regime indicator rather than a blocking gate.

In [12]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets

import diffraction_config as shared_config
import fresnel_config as config
import fresnel_dashboard_tools as dashboard
import fresnel_engine as engine
import fresnel_screen_tools as screen

print("Python:", sys.executable)
for module in (np, plt.matplotlib, widgets):
    print(f"  {module.__name__}: {module.__version__}")
print("Fresnel slit and edge tools loaded OK.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Python: d:\GitHub\Optics\.venv\Scripts\python.exe
  numpy: 2.4.6
  matplotlib: 3.11.0
  ipywidgets: 8.1.8
Fresnel slit and edge tools loaded OK.


## 1. Analytical sanity checks

These checks validate the implementation against the source document before the dashboard is constructed:

1. the finite-source slit example gives $I(0)/I_0\approx1.058$;
2. the finite-source edge example at $u=0.7$ gives $I/I_0\approx0.0665$;
3. every straight edge has $I(0)/I_0=1/4$ at its geometrical boundary;
4. the slit intensity is symmetric around the optical axis.

In [13]:
# PDF finite-source slit example: d=D=1 m, lambda=592 nm, width=0.58 mm.
slit_example = engine.slit_intensity(
    0.0,
    slit_width=0.58e-3,
    wavelength_vacuum=592e-9,
    distance=1.0,
    illumination="point",
    source_distance=1.0,
)
np.testing.assert_allclose(slit_example, 1.0584, rtol=5e-4)

# PDF edge example expressed at dimensionless coordinate u=0.7.
scale = engine.screen_fresnel_scale(
    592e-9, 1.0, illumination="point", source_distance=1.0
)
edge_example = engine.edge_intensity(
    0.7 / scale,
    592e-9,
    1.0,
    illumination="point",
    source_distance=1.0,
)
np.testing.assert_allclose(edge_example, 0.06649, rtol=5e-4)
np.testing.assert_allclose(engine.edge_intensity(0.0, 592e-9, 1.0), 0.25)

coordinates = np.linspace(-3e-3, 3e-3, 301)
slit_profile = engine.slit_intensity(coordinates, 1e-3, 633e-9, 0.5)
np.testing.assert_allclose(slit_profile, slit_profile[::-1], atol=1e-12)

print("All Fresnel analytical sanity checks passed.")

All Fresnel analytical sanity checks passed.


## 2. Adjustable near-field condition

For a slit of full width $b$, this notebook evaluates the user-selected condition

$$N_F=\frac{b^2}{4\lambda D_{\mathrm{eff}}}\ge N_{F,\min},$$

where $\lambda=\lambda_0/n$. For plane-wave illumination, $D_{\mathrm{eff}}=D$; for a finite point source,

$$D_{\mathrm{eff}}=\frac{dD}{d+D}.$$

The **near-field $N_{F,\min}$** control is logarithmic from $10^{-4}$ to $10$, with default $0.1$. It changes only the selected PASS/NOT MET teaching condition. Conventional physical labels remain independent: $N_F<0.1$ is the Fraunhofer limit, $0.1\le N_F<1$ is the Fresnel transition, and $N_F\ge1$ is developed Fresnel near field. The Fresnel solution is always calculated.

A straight edge has infinite transverse support, so no finite aperture Fresnel number exists. The threshold control is disabled in Edge mode, whose status instead reports the characteristic screen length $\ell_F$ corresponding to $\Delta u=1$.

In [14]:
# Evaluate the default slit against the configurable teaching criterion.
default_report = engine.evaluate_fresnel_regime(
    "slit",
    wavelength_vacuum=shared_config.WAVELENGTH_NM.to_si(
        shared_config.WAVELENGTH_NM.default
    ),
    distance=config.DISTANCE_M.default,
    slit_width=config.SLIT_WIDTH_MM.to_si(config.SLIT_WIDTH_MM.default),
    n=shared_config.REFRACTIVE_INDEX.default,
    illumination=config.DEFAULT_ILLUMINATION,
    near_field_limit=config.NEAR_FIELD_LIMIT.default,
)
assert default_report.fresnel_number is not None
assert default_report.is_near_field

print(
    f"Default slit: N_F={default_report.fresnel_number:.3f}, "
    f"selected N_F,min={default_report.near_field_limit:g} -> PASS; "
    f"physical label: {default_report.regime}.\n"
    f"Dashboard threshold range: {config.NEAR_FIELD_LIMIT.minimum:g} "
    f"to {config.NEAR_FIELD_LIMIT.maximum:g}."
)

Default slit: N_F=0.790, selected N_F,min=0.1 -> PASS; physical label: Fresnel transition.
Dashboard threshold range: 0.0001 to 10.


## 3. Interactive dashboard

The Fresnel workbench follows the Fraunhofer notebook's UI/UX while using independent near-field physics:

1. Choose **Slit** or **Edge** in the aperture card.
2. Select **Plane wave** or **Point source** illumination. Point-source mode adds source distance $d$ and uses the effective propagation distance $dD/(d+D)$.
3. Slits support vertical/horizontal orientation and physical width. Edges support orientation and the positive/negative shadow side.
4. The camera is wavelength-colored; screen half-width, zoom, resolution, and secondary-maxima boost affect only observation sampling/presentation.
5. Open **Advanced sampling** to vary the logarithmic **near-field $N_{F,\min}$** criterion across $10^{-4}$–$10$. The status reports PASS/NOT MET alongside the independent physical regime.
6. **Profiles** shows physical $I/I_0$. **Cornu spiral** adds an adjustable screen probe and displays the chord responsible for that intensity.
7. The Fresnel solution remains available for every selected threshold; Edge mode disables the finite-aperture criterion.

Use **Auto update**, **Refresh**, and **Reset** in the same way as the Fraunhofer dashboard.

## Dashboard implementation

The equations live in [fresnel_engine.py](fresnel_engine.py), observation sampling in [fresnel_screen_tools.py](fresnel_screen_tools.py), visual helpers in [fresnel_dashboard_tools.py](fresnel_dashboard_tools.py), and the complete widget dashboard in [fresnel_style.py](fresnel_style.py). The next cell remains a minimal launcher.

In [ ]:
import fresnel_style

fresnel_dashboard = fresnel_style.FresnelDashboard()
fresnel_dashboard.show()

### Suggested experiments

- Use a 1 mm slit at 633 nm and shorten $D$ until several internal Fresnel oscillations appear.
- Switch between plane-wave and point-source illumination, then vary $d$ to observe geometrical magnification and the change in effective distance.
- In Edge mode, verify $I/I_0=0.25$ at the boundary and reverse the shadow side to mirror the pattern.
- Rotate either case to confirm that the one-dimensional pattern follows the axis perpendicular to the slit or edge.
- Enable **Cornu spiral**, move the probe through a maximum and minimum, and compare chord length with the intensity profile.
- Increase $D$ until the slit reports the far-field limit, then compare its normalized profile with the Fraunhofer notebook.